In [1]:
%pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10906293 sha256=5e30c0b497e36c2c552803cb9ed5c472efae2e85b2c17afc3efac56edb880b65
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.3 MB/s eta 0:00:00


In [ ]:
%pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 3.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np
import os
import implicit as imp
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import tensorflow_recommenders as tfrs
import tensorflow as tf


In [ ]:
#Loading the data
mbr_data = load_dataset('recmeapp/mobilerec', data_dir='interactions')
mbr_meta = load_dataset('recmeapp/mobilerec', data_dir='app_meta')

# Save dataset to .csv file for creating pandas dataframe
mbr_data['train'].to_csv('./mbr_data.csv', index=False)
mbr_meta['train'].to_csv('./mbr_meta.csv', index=False)

# Convert to pandas dataframe
mobilerec_df = pd.read_csv('./mbr_data.csv')
mobilerec_mf= pd.read_csv('./mbr_meta.csv')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

interactions/mobilerec_final.csv:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

app_meta/app_meta.csv:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/19298 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

In [ ]:
#Take a sample of the data
data_train, data_test = train_test_split(mobilerec_df, test_size=0.33, random_state=42)
mobilerec_df_sample = data_train.sample(n=100, random_state=42).reset_index(drop=True)

In [ ]:
# How many interactions are there in the MobileRec dataset?
#print(f'There are {len(mobilerec_df)} interactions in mobilerec dataset.')

# How many unique app_packages (apps or items) are there?
#print(f'There are {len(mobilerec_df["app_package"].unique())} unique apps in mobilerec dataset.')

# How many unique users are there in the mobilerec dataset?
#print(f'There are {len(mobilerec_df["uid"].unique())} unique users in mobilerec dataset.')

# How many categoris are there?
#print(f'There are {len(mobilerec_df["app_category"].unique())} unique categories in mobilerec dataset.')

#unique users in database
#print(f'There are {len(mobilerec_df_sample["uid"].unique())} unique users in mobilerec dataset.')

#print(f'There are {len(mobilerec_df_sample["app_package"].unique())} unique apps in mobilerec dataset.')

#shape of matrix
#mobilerec_df.shape

# print the names of the unique users
#print(mobilerec_df_sample.iloc[:,5])

#print(mobilerec_df_sample)

In [ ]:
# Get the list of unique user IDs from the sample
sample_uids = mobilerec_df_sample['uid'].unique()

# Filter the original DataFrame to get all interactions for these users
interactions_of_sample_users = data_train[data_train['uid'].isin(sample_uids)]

# Display the app packages interacted with by these users
apps_interacted = interactions_of_sample_users['app_package']
print(len(sample_uids))
print(len(apps_interacted))
print("Apps interacted with by the sample users:")
display(apps_interacted)


100
5565
Apps interacted with by the sample users:


,app_package
10099491,de.lotum.whatsinthefoto.us
7853271,com.starbucks.mobilecard
13650559,coloring.color.number.happy.paint.art.drawing....
8204548,com.playgendary.tanks
9368323,com.gf.palacem4glgl.hwyad.google
...,...
2185454,com.tv5monde.apprendre
10405945,com.firstlinemedical.optum_flm_app
18707656,com.zynga.pottermatch
2891912,com.sendo


In [ ]:
# Count the occurrences of each app package
unique_apps = interactions_of_sample_users['app_package'].unique()
app_counts = apps_interacted.value_counts()
display(app_counts)


,count
app_package,
ai.replika.app,8
com.instagram.android,7
video.like,7
com.tw.tycoon.casino,6
com.roku.remote,6
...,...
com.grapefrukt.games.bore,1
com.orange.magic.board.doodle.coloring,1
com.gameduell.gin.rummy.card.game,1


In [ ]:
# Create mappings for users and apps to integer indices
user_to_index = {uid: i for i, uid in enumerate(sample_uids)}
app_to_index = {app: i for i, app in enumerate(unique_apps)} # Use index for app names

# Create an inverse mapping from index to app package name
index_to_app = {i: app for app, i in app_to_index.items()}

# Create an inverse mapping from index to user package name
index_to_user = {i: app for app, i in user_to_index.items()}

# Group by user and app to get unique interactions and set the value to 1
grouped_interactions = interactions_of_sample_users.groupby(['uid', 'app_package']).size().reset_index(name='count')
#grouped_interactions['count'] = 1 # Set the value to 1 for each unique interaction

# Prepare data for the sparse matrix
data = grouped_interactions['count']
rows = grouped_interactions['uid'].map(user_to_index)
cols = grouped_interactions['app_package'].map(app_to_index)

# Create the COO sparse matrix
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(len(sample_uids), len(unique_apps)))
# Convert the sparse matrix to a dense NumPy array to visualize matrix
dense_matrix = user_item_matrix.toarray()
print("Dense User-Item Interaction Matrix:")
display(dense_matrix)


Dense User-Item Interaction Matrix:


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
# Instantiate the ALS model
# We'll use default parameters for now, but you can tune these later
model = imp.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)

# The implicit library works best with a CSR matrix
# We'll convert the COO matrix to CSR format
user_item_matrix_csr = user_item_matrix.tocsr()

# Train the model
# The 'fit' method expects the user-item matrix
model.fit(user_item_matrix_csr)

print("WALS (ALS) model trained successfully!")
# You can now access the user and item factors like this:
user_factors = model.user_factors
item_factors = model.item_factors


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

WALS (ALS) model trained successfully!


In [ ]:
#find recommendations for a user
user_id = 4
ids, scores = model.recommend(user_id, user_item_matrix_csr[user_id], N=10, filter_already_liked_items=True)

# Get the recommended app package names using the inverse mapping
recommended_apps = [index_to_app[id] for id in ids]

# Check which of the recommended items were already liked by the user (if any)
already_liked = np.isin(ids, user_item_matrix_csr[user_id].indices)


pd.DataFrame({"app_package": recommended_apps, "score": scores, "already_liked": already_liked})

,app_package,score,already_liked
0,com.opera.mini.native.beta,0.098343,False
1,com.wondershare.videap,0.095270,False
2,com.gamegos.mobile.manorcafe,0.081863,False
3,com.skillshare.Skillshare,0.077010,False
4,com.cookapps.redesignstar.homemakeover,0.067034,False
5,com.vizorapps.klondike,0.054602,False
6,com.nexstreaming.app.kinemasterfree,0.053773,False
7,com.activision.callofduty.shooter,0.053508,False
8,com.moez.QKSMS,0.051124,False
9,com.kroger.mobile,0.050875,False


In [ ]:
# Filter the mobilerec_mf DataFrame to include only the apps present in the sample user interactions
apps_meta = mobilerec_mf[mobilerec_mf['app_package'].isin(unique_apps)].reset_index(drop=True)

# Extract the app descriptions
app_descriptions = apps_meta['description']

#transform descriptions into embeddings
model_text = SentenceTransformer("all-MiniLM-L6-v2")
item_text_embeddings = model_text.encode(
    app_descriptions,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
num_users = len(user_to_index)
num_items = len(app_to_index)

user_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_users,
    output_dim=user_factors.shape[1],
    weights=[user_factors],
    trainable=False
)

item_cf_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_items,
    output_dim=item_factors.shape[1],
    weights=[item_factors],
    trainable=False
)

print(item_cf_embedding_layer)

In [ ]:
# Define the text projection layer
text_projection = tf.keras.layers.Dense(units=item_factors.shape[1]) # Project text embeddings to the same dimension as item CF embeddings

# Define the fusion layer (e.g., using a Dense layer after concatenation)
fusion_layer = tf.keras.layers.Dense(units=item_factors.shape[1], activation='relu') # Example fusion layer


def user_model(user_idx):
    return user_embedding_layer(user_idx)

def item_model(item_idx, text_emb):
    cf_emb = item_cf_embedding_layer(item_idx)
    if len(cf_emb.shape) == 1:
        cf_emb = tf.expand_dims(cf_emb, axis=0)
    text_proj = text_projection(text_emb)
    concat = tf.concat([cf_emb, text_proj], axis=-1)
    fused_emb = fusion_layer(concat)
    return tf.nn.l2_normalize(fused_emb, axis=1)

In [ ]:
 # Choose an example item index (e.g., item with index 4)
example_item_index = tf.constant(5)

# Get the corresponding text embedding and add a batch dimension
example_text_embedding = tf.expand_dims(item_text_embeddings[example_item_index], axis=0)

# Pass the example item index and text embedding to the item model
example_fused_embedding = item_model(example_item_index, example_text_embedding)

print(f"Fused embedding for item index {example_item_index.numpy()}:")
display(example_fused_embedding)

Fused embedding for item index 5:


<tf.Tensor: shape=(1, 50), dtype=float32, numpy=
array([[0.01803058, 0.46589315, 0.        , 0.        , 0.08077756,
        0.29004985, 0.11841361, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.01153524, 0.02363732,
        0.        , 0.        , 0.1949152 , 0.04689553, 0.        ,
        0.        , 0.        , 0.0467591 , 0.00938847, 0.30381328,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.44783995, 0.03335878, 0.        , 0.17762928, 0.        ,
        0.28589723, 0.        , 0.36768743, 0.15033905, 0.        ,
        0.        , 0.        , 0.25529182, 0.        , 0.        ,
        0.        , 0.06745472, 0.        , 0.        , 0.        ]],
      dtype=float32)>

In [ ]:
class FunctionalRetrievalModel(tfrs.models.Model):
    def __init__(self, user_model_fn, item_model_fn, item_candidates):
        super().__init__()
        self.user_model_fn = user_model_fn
        self.item_model_fn = item_model_fn
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=item_candidates.batch(128).map(
                    lambda x: item_model_fn(x["item_idx"], x["text_emb"])
                )
            )
        )

    def compute_loss(self, features, training = False):

      user_embeddings = self.user_model_fn(features["user_idx"])
      item_embeddings = self.item_model_fn(features["item_idx"], features["text_emb"])

      return self.task(user_embeddings, item_embeddings)


In [ ]:
def map_interaction_to_tensors(x):
    user_idx = tf.py_function(lambda uid: user_to_index[uid.numpy().decode('utf-8')], [x["uid"]], tf.int64)
    item_idx = tf.py_function(lambda app_package: app_to_index[app_package.numpy().decode('utf-8')], [x["app_package"]], tf.int64)
    text_emb = tf.py_function(lambda app_package: item_text_embeddings[app_to_index[app_package.numpy().decode('utf-8')]], [x["app_package"]], tf.float32)

    # Ensure the shapes are defined
    user_idx.set_shape([])
    item_idx.set_shape([])
    text_emb.set_shape(item_text_embeddings.shape[1:])  # Set the shape based on the actual embedding dimension

    return {
        "user_idx": user_idx,
        "item_idx": item_idx,
        "text_emb": text_emb
    }

def map_candidate_to_tensors(x):
    item_idx = tf.py_function(lambda app_package: app_to_index[app_package.numpy().decode('utf-8')], [x["app_package"]], tf.int64)
    text_emb = tf.py_function(lambda app_package: item_text_embeddings[app_to_index[app_package.numpy().decode('utf-8')]], [x["app_package"]], tf.float32)

    # Ensure the shapes are defined
    item_idx.set_shape([])
    text_emb.set_shape(item_text_embeddings.shape[1:])  # Set the shape based on the actual embedding dimension

    return {
        "item_idx": item_idx,
        "text_emb": text_emb
    }

train_ds = tf.data.Dataset.from_tensor_slices(dict(grouped_interactions)).map(
    map_interaction_to_tensors,
    num_parallel_calls=tf.data.AUTOTUNE
).cache().shuffle(len(grouped_interactions)).batch(4096).prefetch(tf.data.AUTOTUNE)

candidate_ds = tf.data.Dataset.from_tensor_slices(dict(apps_meta)).map(
    map_candidate_to_tensors,
    num_parallel_calls=tf.data.AUTOTUNE
).cache()

In [ ]:
model = FunctionalRetrievalModel(user_model, item_model, candidate_ds)
model.compile(optimizer=tf.keras.optimizers.Adagrad(0.05))
model.fit(train_ds, epochs=5)


Epoch 1/5
2/2 [==============================] - 17s 1s/step - factorized_top_k/top_1_categorical_accuracy: 3.6278e-04 - factorized_top_k/top_5_categorical_accuracy: 9.0695e-04 - factorized_top_k/top_10_categorical_accuracy: 0.0038 - factorized_top_k/top_50_categorical_accuracy: 0.0131 - factorized_top_k/top_100_categorical_accuracy: 0.0294 - loss: 18611.9160 - regularization_loss: 0.0000e+00 - total_loss: 18611.9160
Epoch 2/5
2/2 [==============================] - 4s 1s/step - factorized_top_k/top_1_categorical_accuracy: 3.6278e-04 - factorized_top_k/top_5_categorical_accuracy: 9.0695e-04 - factorized_top_k/top_10_categorical_accuracy: 0.0038 - factorized_top_k/top_50_categorical_accuracy: 0.0131 - factorized_top_k/top_100_categorical_accuracy: 0.0294 - loss: 18616.3750 - regularization_loss: 0.0000e+00 - total_loss: 18616.3750
Epoch 3/5
2/2 [==============================] - 4s 1s/step - factorized_top_k/top_1_categorical_accuracy: 3.6278e-04 - factorized_top_k/top_5_categorical_accu

In [ ]:
# Suppose you have a new app text embedding
text_dim = item_text_embeddings.shape[1]
new_text_emb = tf.random.normal((1, text_dim))
new_item_idx = tf.constant([5])  # existing or dummy index

# Compute item embedding
new_item_emb = item_model(new_item_idx, new_text_emb)

# Compute similarities manually (for now)
user_embeddings = user_model(tf.range(num_users))
scores = tf.linalg.matmul(new_item_emb, user_embeddings, transpose_b=True)
top_users = tf.argsort(scores, direction="DESCENDING")[0, :5]
print("Top user candidates:", top_users.numpy())


Top user candidates: [40 98 53 85 96]
